# MOT with pMOT Fields trajectory lab

This notebook runs the physical Section-12 24-state multilevel MOT with its six cooling beams, six repump beams, deterministic mean radiation pressure, and gravity. It does **not** use an anti-Helmholtz coil. Instead, the intended 1529/1530-nm trapping-beam intensity and helicity geometry is mapped to a stretched-transition-equivalent surrogate magnetic field and supplied to the MOT Zeeman calculation at every force evaluation and RK4 stage.

Edit the `PHYSICS`, `SURROGATE_FIELD`, and `SIMULATION` dictionaries, then run all cells. The notebook supports the same manual, exact direction-disc, and seeded full-sphere launch modes as `configured_trajectory_lab.ipynb`.

This is a **MOT with pMOT Fields**, not a physical pMOT simulation. The trapping light enters only through the surrogate magnetic field; the state-resolved AC-Stark Hamiltonian, conservative trapping-light force, trapping-light scattering/heating, coherent interference, measured polarization transformations, and nonadiabatic dynamics at the field zero are not included.

## 1. Imports and model initialization

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace
from datetime import datetime
import json
from time import perf_counter

from matplotlib import animation as mpl_animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from pmot.configuration import default_mot_apparatus_config
from pmot.launch_geometry import build_incident_disc_from_angles
from pmot.mot_multilevel import (
    CaptureSearchConfig,
    RateEquationAtomState,
    RateEquationTrajectoryConfig,
    build_multilevel_mot_beams,
    build_rate_equation_model,
    create_trajectory_animation,
    default_multilevel_mot_config,
    generate_capture_launches,
    multilevel_mot_paths,
    plot_time_diagnostics,
    plot_trajectory_3d,
    save_trajectory,
    simulate_rate_equation_trajectory,
    trajectory_summary,
)
from pmot.pmot.surrogate_effective_field import (
    SurrogateEffectiveFieldConfig,
    inside_surrogate_cell,
    numerical_field_jacobian_t_per_m,
    surrogate_effective_field_t,
)
from pmot.pmot.surrogate_effective_field_study import classify_diagnostic_trajectory

MODEL = build_rate_equation_model()
PROJECT_ROOT = multilevel_mot_paths()['root']
print(f'Loaded {MODEL.state_count} states and {MODEL.transition_count} allowed transitions.')

## 2. MOT and surrogate-field configuration

Cooling and repump powers are per traveling beam. The surrogate path powers and signed helicity weights define the effective-field model; they are not a substitute for laboratory polarization transformations.

In [ ]:
PHYSICS = {
    'cooling_power_mw_per_beam': 27.0,
    'repump_power_mw_per_beam': 0.1,
    'cooling_detuning_mhz': -15.0,
    'repump_detuning_mhz': 0.0,
    'cooling_wavelength_nm': 780.0,
    'repump_wavelength_nm': 780.232684,
    'cooling_beam_diameter_mm': 12.7,
    'repump_beam_diameter_mm': 12.7,
    'repumper_enabled': True,
    'include_gravity': True,
}

SURROGATE_FIELD = {
    # Center-balanced optical-envelope geometry
    'wavelength_nm': 1530.0,
    'forward_waist_position_mm': -10.0326971,
    'forward_rayleigh_range_um': 9.103072091279295,
    'return_waist_position_mm': 9.97601576,
    'return_rayleigh_range_um': 9.000504018534303,

    # Configured power on each Cartesian round-trip path
    'x_path_power_mw': 10.8,
    'y_path_power_mw': 10.8,
    'z_path_power_mw': 21.6,

    # Signed component weights used by the surrogate-field mapping
    'helicity_positive_x': 1.0,
    'helicity_negative_x': 1.0,
    'helicity_positive_y': 1.0,
    'helicity_negative_y': 1.0,
    'helicity_positive_z': -1.0,
    'helicity_negative_z': -1.0,

    # Vapor-cell boundary and stretched-transition conversion
    'cell_half_length_mm': 14.1421356,
    'transition_zeeman_factor': 1.0,
    'differential_vector_polarizability_hz_per_w_per_m2': 1363.34,
}

display(pd.Series(PHYSICS, name='value').to_frame())
display(pd.Series(SURROGATE_FIELD, name='value').to_frame())

## 3. Numerical and launch configuration

Choose `manual`, `specified_disc`, or `random_full_sphere`. In `specified_disc`, the polar and azimuthal angles locate the sampling-disc center on the launch sphere; the velocity points inward along the opposite normal. `impact_azimuth_deg` is required to orient a nonzero impact parameter within the disc plane.

In [ ]:
SIMULATION = {
    # Integration
    'duration_ms': 10.0,
    'time_step_us': 5.0,
    'escape_radius_mm': 30.0,

    # Select 'manual', 'specified_disc', or 'random_full_sphere'
    'launch_mode': 'specified_disc',
    'trajectory_count': 1,
    'random_seed': 20261002,

    # Shared direction-disc speed and radius
    'launch_radius_mm': 12.0,
    'launch_speed_m_per_s': 5.0,

    # Seeded full-sphere mode only
    'impact_disc_radius_mm': 2.0,
    'launch_speed_std_m_per_s': 0.0,

    # Exact direction-disc mode only
    'launch_polar_angle_deg': 90.0,
    'launch_azimuthal_angle_deg': 0.0,
    'impact_parameter_mm': 0.0,
    'impact_azimuth_deg': 0.0,

    # Manual Cartesian mode only
    'manual_position_mm': (12.0, 0.0, 0.0),
    'manual_velocity_m_per_s': (-5.0, 0.0, 0.0),

    # Visualization and persistence
    'field_plot_extent_mm': 14.0,
    'field_plot_samples_per_axis': 501,
    'field_plot_symlog_linthresh_g': 1.0e-3,
    'trajectory_plot_extent_mm': 15.0,
    'selected_trajectory_index': 0,
    'animation_max_frames': 160,
    'animation_fps': 20,
    'save_results': False,
    'save_animation_gif': False,
    'run_name': 'mot_with_pmot_fields_trajectory_lab',
}

pd.Series(SIMULATION, name='value').to_frame()

## 4. Build and review the configuration

In [ ]:
def _positive(mapping, keys):
    for key in keys:
        if float(mapping[key]) <= 0.0:
            raise ValueError(f'{key} must be positive')


def build_notebook_configuration(physics, field, simulation):
    _positive(physics, [
        'cooling_wavelength_nm', 'repump_wavelength_nm',
        'cooling_beam_diameter_mm', 'repump_beam_diameter_mm',
    ])
    _positive(field, [
        'wavelength_nm', 'forward_rayleigh_range_um',
        'return_rayleigh_range_um', 'cell_half_length_mm',
        'transition_zeeman_factor',
        'differential_vector_polarizability_hz_per_w_per_m2',
    ])
    _positive(simulation, [
        'duration_ms', 'time_step_us', 'escape_radius_mm',
        'launch_radius_mm', 'launch_speed_m_per_s',
        'field_plot_extent_mm', 'field_plot_samples_per_axis',
        'field_plot_symlog_linthresh_g',
        'trajectory_plot_extent_mm', 'animation_max_frames', 'animation_fps',
    ])
    if physics['cooling_power_mw_per_beam'] < 0 or physics['repump_power_mw_per_beam'] < 0:
        raise ValueError('MOT beam powers must be non-negative')
    if min(field['x_path_power_mw'], field['y_path_power_mw'], field['z_path_power_mw']) < 0:
        raise ValueError('surrogate path powers must be non-negative')
    if min(simulation['impact_disc_radius_mm'], simulation['impact_parameter_mm'], simulation['launch_speed_std_m_per_s']) < 0:
        raise ValueError('disc radius, impact parameter, and speed spread must be non-negative')
    if simulation['launch_mode'] not in {'manual', 'specified_disc', 'random_full_sphere'}:
        raise ValueError("launch_mode must be 'manual', 'specified_disc', or 'random_full_sphere'")
    if not 0.0 <= float(simulation['launch_polar_angle_deg']) <= 180.0:
        raise ValueError('launch_polar_angle_deg must lie between 0 and 180 degrees')
    if int(simulation['trajectory_count']) < 1:
        raise ValueError('trajectory_count must be at least one')

    rate_config = replace(
        default_multilevel_mot_config(),
        cooling_detuning_rad_per_s=2.0 * np.pi * physics['cooling_detuning_mhz'] * 1.0e6,
        repump_detuning_rad_per_s=2.0 * np.pi * physics['repump_detuning_mhz'] * 1.0e6,
        cooling_power_w_per_beam=physics['cooling_power_mw_per_beam'] * 1.0e-3,
        repump_power_w_per_beam=physics['repump_power_mw_per_beam'] * 1.0e-3,
        cooling_wavelength_m=physics['cooling_wavelength_nm'] * 1.0e-9,
        repump_wavelength_m=physics['repump_wavelength_nm'] * 1.0e-9,
        repumper_enabled=bool(physics['repumper_enabled']),
        include_gravity=bool(physics['include_gravity']),
    )
    base_apparatus = default_mot_apparatus_config()
    apparatus = replace(
        base_apparatus,
        cooling=replace(base_apparatus.cooling, beam_diameter_m=physics['cooling_beam_diameter_mm'] * 1.0e-3),
        repump=replace(base_apparatus.repump, beam_diameter_m=physics['repump_beam_diameter_mm'] * 1.0e-3),
    )
    beams = build_multilevel_mot_beams(apparatus_config=apparatus, config=rate_config)
    field_config = SurrogateEffectiveFieldConfig(
        wavelength_m=field['wavelength_nm'] * 1.0e-9,
        forward_waist_position_m=field['forward_waist_position_mm'] * 1.0e-3,
        forward_rayleigh_range_m=field['forward_rayleigh_range_um'] * 1.0e-6,
        return_waist_position_m=field['return_waist_position_mm'] * 1.0e-3,
        return_rayleigh_range_m=field['return_rayleigh_range_um'] * 1.0e-6,
        x_path_power_w=field['x_path_power_mw'] * 1.0e-3,
        y_path_power_w=field['y_path_power_mw'] * 1.0e-3,
        z_path_power_w=field['z_path_power_mw'] * 1.0e-3,
        helicity_positive_x=field['helicity_positive_x'],
        helicity_negative_x=field['helicity_negative_x'],
        helicity_positive_y=field['helicity_positive_y'],
        helicity_negative_y=field['helicity_negative_y'],
        helicity_positive_z=field['helicity_positive_z'],
        helicity_negative_z=field['helicity_negative_z'],
        cell_half_length_m=field['cell_half_length_mm'] * 1.0e-3,
        transition_zeeman_factor=field['transition_zeeman_factor'],
        differential_vector_polarizability_hz_per_w_per_m2=field['differential_vector_polarizability_hz_per_w_per_m2'],
    )
    numerical = RateEquationTrajectoryConfig(
        time_step_s=simulation['time_step_us'] * 1.0e-6,
        escape_radius_m=simulation['escape_radius_mm'] * 1.0e-3,
    )

    launch_metadata = []
    if simulation['launch_mode'] == 'manual':
        position = tuple(float(value) * 1.0e-3 for value in simulation['manual_position_mm'])
        velocity = tuple(float(value) for value in simulation['manual_velocity_m_per_s'])
        initial_states = [RateEquationAtomState(position, velocity)]
        launch_metadata.append({'mode': 'manual', 'position_m': position, 'velocity_m_per_s': velocity})
    elif simulation['launch_mode'] == 'specified_disc':
        polar_rad = float(np.deg2rad(simulation['launch_polar_angle_deg']))
        azimuthal_rad = float(np.deg2rad(simulation['launch_azimuthal_angle_deg']))
        impact_azimuth_rad = float(np.deg2rad(simulation['impact_azimuth_deg']))
        impact_parameter_m = float(simulation['impact_parameter_mm']) * 1.0e-3
        disc = build_incident_disc_from_angles(
            0, simulation['launch_radius_mm'] * 1.0e-3, polar_rad, azimuthal_rad
        )
        offset = impact_parameter_m * (
            np.cos(impact_azimuth_rad) * np.asarray(disc.basis_u)
            + np.sin(impact_azimuth_rad) * np.asarray(disc.basis_v)
        )
        position = tuple(float(value) for value in np.asarray(disc.center_position_m) + offset)
        velocity = tuple(
            float(value) for value in simulation['launch_speed_m_per_s'] * np.asarray(disc.incident_unit_vector)
        )
        initial_states = [RateEquationAtomState(position, velocity)]
        launch_metadata.append({
            'mode': 'specified_disc', 'theta_rad': polar_rad, 'phi_rad': azimuthal_rad,
            'impact_azimuth_rad': impact_azimuth_rad, 'impact_radius_m': impact_parameter_m,
            'position_m': position, 'velocity_m_per_s': velocity,
        })
    else:
        requested_radius_m = simulation['impact_disc_radius_mm'] * 1.0e-3
        search = CaptureSearchConfig(
            radial_distance_m=simulation['launch_radius_mm'] * 1.0e-3,
            disc_radius_m=max(requested_radius_m, np.finfo(float).eps),
            disc_count=int(simulation['trajectory_count']), points_per_disc=1,
            include_center_point=(requested_radius_m == 0.0),
            maximum_simulation_time_s=simulation['duration_ms'] * 1.0e-3,
            time_step_s=simulation['time_step_us'] * 1.0e-6,
            escape_radius_m=simulation['escape_radius_mm'] * 1.0e-3,
            seed=int(simulation['random_seed']),
        )
        _, points = generate_capture_launches(search)
        speed_rng = np.random.default_rng(int(simulation['random_seed']) + 1)
        initial_states = []
        for point in points:
            speed = float(simulation['launch_speed_m_per_s'])
            if simulation['launch_speed_std_m_per_s'] > 0.0:
                speed = max(0.0, float(speed_rng.normal(speed, simulation['launch_speed_std_m_per_s'])))
            velocity = tuple(float(value) for value in speed * np.asarray(point.incident_unit_vector))
            initial_states.append(RateEquationAtomState(point.initial_position_m, velocity))
            launch_metadata.append({
                'mode': 'random_full_sphere', 'disc_index': point.disc_index,
                'theta_rad': point.theta_rad, 'phi_rad': point.phi_rad,
                'impact_radius_m': point.s_m if requested_radius_m > 0.0 else 0.0,
                'position_m': point.initial_position_m, 'velocity_m_per_s': velocity,
            })
    return rate_config, apparatus, beams, field_config, numerical, initial_states, launch_metadata


RATE_CONFIG, APPARATUS, BEAMS, FIELD_CONFIG, NUMERICAL, INITIAL_STATES, LAUNCH_METADATA = (
    build_notebook_configuration(PHYSICS, SURROGATE_FIELD, SIMULATION)
)

def effective_field(position_m):
    return surrogate_effective_field_t(position_m, FIELD_CONFIG)

def inside_field_domain(position_m):
    return inside_surrogate_cell(position_m, FIELD_CONFIG)

origin_field_g = 1.0e4 * effective_field((0.0, 0.0, 0.0))
origin_jacobian_g_per_cm = 100.0 * numerical_field_jacobian_t_per_m(config=FIELD_CONFIG)
launch_table = pd.DataFrame(LAUNCH_METADATA)
launch_table['inside_cell'] = [inside_field_domain(state.position_m) for state in INITIAL_STATES]
display(launch_table)
print(f'Origin effective field [G]: {origin_field_g}')
print('Origin field Jacobian [G/cm]:')
display(pd.DataFrame(origin_jacobian_g_per_cm, index=['Bx', 'By', 'Bz'], columns=['x', 'y', 'z']))
print(f'Trajectories: {len(INITIAL_STATES)}')
print(f'Nominal RK4 steps per full trajectory: {int(np.ceil(SIMULATION["duration_ms"] * 1000 / SIMULATION["time_step_us"])):,}')

## 5. Inspect the surrogate magnetic-field shape

The signed field component reverses at the trap center, so these plots use a symmetric logarithmic y-axis. Values outside the small editable linear interval around zero are logarithmic, while the sign reversal and exact zero remain visible.

In [ ]:
extent_m = SIMULATION['field_plot_extent_mm'] * 1.0e-3
coordinate_m = np.linspace(-extent_m, extent_m, int(SIMULATION['field_plot_samples_per_axis']))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for axis_index, (axis_name, axis) in enumerate(zip('xyz', axes)):
    points = np.zeros((len(coordinate_m), 3))
    points[:, axis_index] = coordinate_m
    fields_g = 1.0e4 * surrogate_effective_field_t(points, FIELD_CONFIG)
    axis.plot(1.0e3 * coordinate_m, fields_g[:, axis_index], color='#7c3aed', label=f'B{axis_name}')
    axis.plot(1.0e3 * coordinate_m, np.linalg.norm(fields_g, axis=1), color='#0f766e', alpha=0.75, label='|B|')
    axis.axhline(0.0, color='#64748b', linewidth=0.8)
    axis.set_yscale('symlog', linthresh=SIMULATION['field_plot_symlog_linthresh_g'])
    axis.set(title=f'{axis_name}-axis field cut', xlabel=f'{axis_name} [mm]', ylabel='Effective field [G] (symmetric log)')
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle('Centered surrogate magnetic field generated from intended trapping-beam envelopes')
plt.show()

## 6. Run the trajectories

In [ ]:
safe_run_name = ''.join(
    character if character.isalnum() or character in '-_' else '_'
    for character in SIMULATION['run_name']
).strip('_')
if not safe_run_name:
    raise ValueError('run_name must contain at least one letter or number')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
trajectory_root = PROJECT_ROOT / 'outputs' / 'trajectories' / 'mot_with_pmot_fields' / f'{safe_run_name}_{timestamp}'
figure_root = PROJECT_ROOT / 'outputs' / 'figures' / 'mot_with_pmot_fields' / f'{safe_run_name}_{timestamp}'

RECORDS = []
SUMMARY_ROWS = []
ensemble_started = perf_counter()
for index, initial_state in enumerate(INITIAL_STATES):
    started = perf_counter()
    print(f'Running trajectory {index + 1}/{len(INITIAL_STATES)} ...', flush=True)
    record = simulate_rate_equation_trajectory(
        initial_state,
        SIMULATION['duration_ms'] * 1.0e-3,
        coil_config=None,
        beams=BEAMS, model=MODEL, config=RATE_CONFIG, trajectory_config=NUMERICAL,
        magnetic_field_function=effective_field,
        spatial_domain_function=inside_field_domain,
    )
    wall_time = perf_counter() - started
    classification = classify_diagnostic_trajectory(record)
    RECORDS.append(record)
    SUMMARY_ROWS.append({
        'trajectory_index': index, 'wall_time_s': wall_time,
        **trajectory_summary(record), **classification,
    })
    print(
        f"  termination={record.termination_reason}; classification={classification['classification']}; "
        f"elapsed={1e3 * record.times_s[-1]:.3f} ms; wall={wall_time:.2f} s"
    )
    if SIMULATION['save_results']:
        save_trajectory(
            record, MODEL, BEAMS, trajectory_root / f'trajectory_{index:04d}',
            metadata={
                'model': 'MOT with pMOT Fields',
                'interpretation': 'surrogate-field MOT diagnostic; not a physical pMOT',
                'physics_parameters': PHYSICS, 'surrogate_field_parameters': SURROGATE_FIELD,
                'simulation_parameters': SIMULATION, 'rate_config': asdict(RATE_CONFIG),
                'apparatus_config': asdict(APPARATUS), 'field_config': FIELD_CONFIG.metadata(),
                'launch': LAUNCH_METADATA[index], 'trajectory_index': index,
            },
        )

SUMMARY = pd.DataFrame(SUMMARY_ROWS)
print(f'Ensemble wall time: {perf_counter() - ensemble_started:.2f} s')
display(SUMMARY)
if SIMULATION['save_results']:
    trajectory_root.mkdir(parents=True, exist_ok=True)
    SUMMARY.to_csv(trajectory_root / 'ensemble_summary.csv', index=False)
    manifest = {
        'schema': 'pmot.mot-with-pmot-fields-trajectory-notebook.v1',
        'model': 'physical Section-12 MOT with surrogate effective magnetic field',
        'physics_parameters': PHYSICS, 'surrogate_field_parameters': SURROGATE_FIELD,
        'simulation_parameters': SIMULATION, 'field_config': FIELD_CONFIG.metadata(),
        'launches': LAUNCH_METADATA,
    }
    (trajectory_root / 'campaign_manifest.json').write_text(
        json.dumps(manifest, indent=2) + '\n', encoding='utf-8'
    )
    print(f'Saved trajectory data to {trajectory_root}')

## 7. Three-dimensional path, MOT diagnostics, and field history

In [ ]:
if not RECORDS:
    raise RuntimeError('Run the trajectory cell first')
selected_index = int(SIMULATION['selected_trajectory_index'])
if not 0 <= selected_index < len(RECORDS):
    raise IndexError(f'selected_trajectory_index must be between 0 and {len(RECORDS) - 1}')
selected_record = RECORDS[selected_index]

def draw_cell_boundary(axis, config, *, color='#f59e0b'):
    half_mm = 1.0e3 * config.cell_half_length_m
    for first_axis in (-half_mm, half_mm):
        for second_axis in (-half_mm, half_mm):
            axis.plot([-half_mm, half_mm], [first_axis, first_axis], [second_axis, second_axis], color=color, alpha=0.35)
            axis.plot([first_axis, first_axis], [-half_mm, half_mm], [second_axis, second_axis], color=color, alpha=0.35)
            axis.plot([first_axis, first_axis], [second_axis, second_axis], [-half_mm, half_mm], color=color, alpha=0.35)

TRAJECTORY_FIGURE = plot_trajectory_3d(
    selected_record, BEAMS, None,
    title=f'MOT with pMOT Fields trajectory {selected_index}',
    spatial_extent_mm=SIMULATION['trajectory_plot_extent_mm'],
)
draw_cell_boundary(TRAJECTORY_FIGURE.axes[0], FIELD_CONFIG)
DIAGNOSTICS_FIGURE = plot_time_diagnostics(selected_record, MODEL, BEAMS)

times_ms = 1.0e3 * np.asarray(selected_record.times_s)
fields_g = 1.0e4 * surrogate_effective_field_t(np.asarray(selected_record.positions_m), FIELD_CONFIG)
FIELD_HISTORY_FIGURE, field_axis = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
for component, color in zip(range(3), ('#b91c1c', '#1d4ed8', '#15803d')):
    field_axis.plot(times_ms, fields_g[:, component], color=color, label=f'B{"xyz"[component]}')
field_axis.plot(times_ms, np.linalg.norm(fields_g, axis=1), color='#111827', linestyle='--', label='|B|')
field_axis.set(title='Surrogate effective magnetic field along trajectory', xlabel='Time [ms]', ylabel='Field [G]')
field_axis.grid(alpha=0.25)
field_axis.legend()

if SIMULATION['save_results']:
    figure_root.mkdir(parents=True, exist_ok=True)
    TRAJECTORY_FIGURE.savefig(figure_root / f'trajectory_{selected_index:04d}_3d.png', dpi=180, bbox_inches='tight')
    DIAGNOSTICS_FIGURE.savefig(figure_root / f'trajectory_{selected_index:04d}_diagnostics.png', dpi=180, bbox_inches='tight')
    FIELD_HISTORY_FIGURE.savefig(figure_root / f'trajectory_{selected_index:04d}_field_history.png', dpi=180, bbox_inches='tight')
plt.show()

## 8. Coarse-grained full-trajectory animation

In [ ]:
if not RECORDS:
    raise RuntimeError('Run the trajectory cell first')
selected_record = RECORDS[int(SIMULATION['selected_trajectory_index'])]
requested_frames = int(SIMULATION['animation_max_frames'])
print(
    f'Animating up to {min(requested_frames, len(selected_record.times_s))} evenly spaced frames '
    f'across the complete {1e3 * selected_record.times_s[-1]:.3f} ms trajectory.'
)

def build_animation(max_frames):
    movie = create_trajectory_animation(
        selected_record, BEAMS, None, max_frames=max_frames,
        fps=int(SIMULATION['animation_fps']),
        spatial_extent_mm=SIMULATION['trajectory_plot_extent_mm'],
    )
    movie._fig.axes[0].set_title('MOT with pMOT Fields trajectory')
    draw_cell_boundary(movie._fig.axes[0], FIELD_CONFIG)
    return movie

TRAJECTORY_ANIMATION = build_animation(requested_frames)
if SIMULATION['save_animation_gif']:
    figure_root.mkdir(parents=True, exist_ok=True)
    gif_path = figure_root / f'trajectory_{int(SIMULATION["selected_trajectory_index"]):04d}.gif'
    TRAJECTORY_ANIMATION.save(
        gif_path, writer=mpl_animation.PillowWriter(fps=int(SIMULATION['animation_fps'])), dpi=110
    )
    print(f'Saved animation to {gif_path}')
try:
    inline_animation = TRAJECTORY_ANIMATION.to_html5_video()
except Exception as error:
    inline_animation = ''
    print(f'Compressed video encoding unavailable ({error}); using a low-memory HTML fallback.')
if '<video' not in inline_animation:
    plt.close(TRAJECTORY_ANIMATION._fig)
    fallback_frames = min(requested_frames, 40)
    TRAJECTORY_ANIMATION = build_animation(fallback_frames)
    inline_animation = TRAJECTORY_ANIMATION.to_jshtml()
display(HTML(inline_animation))
plt.close(TRAJECTORY_ANIMATION._fig)

## Interpretation boundary

These trajectories test whether the intended pMOT-like field shape produces MOT restoring, damping, and bounded-trajectory behavior when treated as a magnetic field. They use deterministic mean force and do not include recoil diffusion. A visually bounded trajectory is not by itself a converged capture classification, and no result from this notebook should be labeled a physical pMOT prediction.